# 06 — Imputation strategy (data-processing handoff)

This notebook delivers an input to feed the models. Observation about missingness described in `02_missing_values.ipynb`).

We split preprocessing into layers:

1) **Layer A** (`src/features.py`) is deterministic feature engineering
  (drop leakage, merge the age-gated PAQ, add block-missing indicators).
2) **Layer B** (`src/imputation.py`) is reference CV comparison.
  It is fit inside each CV fold (not on the whole dataset), output is not stored.

**Scope decision:** we use 3 tabular CSVs. The actigraphy time
series (`series_*.parquet`) are a separate modality with heavy missingness and
are out of scope for this stage; they cannot be used to fill tabular gaps.

## 1. Imports and data loading

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Bootstrap: make the src package importable whether cwd is the repo root or notebooks/
_ROOT = Path.cwd()
if _ROOT.name == "notebooks":
    _ROOT = _ROOT.parent
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))

# Single source of truth for paths and settings (flat data/ layout, see src/config.py)
from src.config import (
    ID_COLUMN,
    PROCESSED_DIR,
    PROJECT_ROOT,
    RESULTS_DIR,
    TARGET,
    TEST_PATH,
    TRAIN_PATH,
)
from src.evaluation import create_cv_splits, evaluate_model, regression_to_classes
from src.features import build_features
from src.imputation import make_preprocessor

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

print("Train:", train.shape)
print("Test:", test.shape)
print("Labeled train rows:", train[TARGET].notna().sum())

Train: (3960, 82)
Test: (20, 59)
Labeled train rows: 2736


## 2. Layer A

`build_features` first turns EDA findings into cleaning rules, then engineers features:

1. **codebook validation** — values outside the data dictionary's allowed set → NaN;
2. **domain-range validity** — physiologically impossible values (e.g. weight/BMI/BP = 0, CGAS = 999) → NaN;
3. **BMI cross-consistency** — flag stated-vs-recomputed BMI, null gross errors;
4. drop `PCIAT-*` leakage; merge age-gated `PAQ_C`/`PAQ_A` → `PAQ_total` (+ `PAQ_version`, `PAQ_missing`, `PAQ_season`); add `*_not_administered` flags per block.

Steps 1–3 use fixed references (the dictionary, clinical bounds, the BMI formula), not sample statistics, so they stay row-local and safe to persist. The result keeps NaN in sporadic gaps by design.

In [2]:
train_features, validation_report = build_features(train, return_report=True)
test_features = build_features(test)

new_columns = [c for c in train_features.columns if c not in train.columns]
dropped_columns = [c for c in train.columns if c not in train_features.columns]

print("train_features:", train_features.shape)
print("test_features:", test_features.shape)
print("\nColumns added by Layer A:")
print(new_columns)
print("\nColumns removed by Layer A (leakage + merged PAQ):")
print(dropped_columns)

print("\n--- Validation (cells voided to NaN on train) ---")
print("codebook (out-of-allowed-set):", validation_report["codebook"] or "none")
print("domain-range (impossible values):", validation_report["domain"] or "none")
print("BMI consistency (recomputed vs stated):", validation_report["bmi"])

train_features: (3960, 67)
test_features: (20, 66)

Columns added by Layer A:
['bmi_inconsistent', 'PAQ_total', 'PAQ_missing', 'PAQ_version', 'PAQ_season', 'CGAS_not_administered', 'Physical_not_administered', 'Fitness_Endurance_not_administered', 'FGC_not_administered', 'BIA_not_administered', 'SDS_not_administered']

Columns removed by Layer A (leakage + merged PAQ):
['PAQ_A-Season', 'PAQ_A-PAQ_A_Total', 'PAQ_C-Season', 'PAQ_C-PAQ_C_Total', 'PCIAT-Season', 'PCIAT-PCIAT_01', 'PCIAT-PCIAT_02', 'PCIAT-PCIAT_03', 'PCIAT-PCIAT_04', 'PCIAT-PCIAT_05', 'PCIAT-PCIAT_06', 'PCIAT-PCIAT_07', 'PCIAT-PCIAT_08', 'PCIAT-PCIAT_09', 'PCIAT-PCIAT_10', 'PCIAT-PCIAT_11', 'PCIAT-PCIAT_12', 'PCIAT-PCIAT_13', 'PCIAT-PCIAT_14', 'PCIAT-PCIAT_15', 'PCIAT-PCIAT_16', 'PCIAT-PCIAT_17', 'PCIAT-PCIAT_18', 'PCIAT-PCIAT_19', 'PCIAT-PCIAT_20', 'PCIAT-PCIAT_Total']

--- Validation (cells voided to NaN on train) ---
codebook (out-of-allowed-set): none
domain-range (impossible values): {'Physical-BMI': 7, 'Physical-Weigh

In [3]:
# NaN left for Layer B to fill
remaining_nan = (
    train_features.drop(columns=[ID_COLUMN, TARGET])
    .isna()
    .mean()
    .sort_values(ascending=False)
)
print("Features containing NaN:", (remaining_nan > 0).sum())
print("\nTop 10 by remaining NaN fraction:")
print((remaining_nan.head(10) * 100).round(1).to_string())

Features containing NaN: 53

Top 10 by remaining NaN fraction:
Fitness_Endurance-Time_Mins     81.3
Fitness_Endurance-Time_Sec      81.3
Fitness_Endurance-Max_Stage     81.2
Physical-Waist_Circumference    77.3
FGC-FGC_GSND_Zone               73.2
FGC-FGC_GSD_Zone                73.2
FGC-FGC_GSD                     72.9
FGC-FGC_GSND                    72.9
Fitness_Endurance-Season        67.0
BIA-BIA_BMC                     49.7


### Saving the output

Input in `\data\processed` keeps `id` and (for train) `sii`,
includes all 3960 train rows. The fold-safe imputer fills remaining NaN's at modeling time.

In [4]:
from src.features import feature_type_map, load_data_dictionary

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

train_features.to_parquet(PROCESSED_DIR / "train_features.parquet", index=False)
test_features.to_parquet(PROCESSED_DIR / "test_features.parquet", index=False)

# Explicit, dictionary-driven type contract for the modeling team
# (identifier / leakage / categorical / continuous per field).
type_map = feature_type_map(load_data_dictionary())
RESULTS_DIR.mkdir(exist_ok=True)
(
    pd.Series(type_map, name="declared_type")
    .rename_axis("field")
    .to_frame()
    .to_csv(RESULTS_DIR / "feature_types.csv")
)

print("Saved:")
print(" ", (PROCESSED_DIR / "train_features.parquet").relative_to(PROJECT_ROOT))
print(" ", (PROCESSED_DIR / "test_features.parquet").relative_to(PROJECT_ROOT))
print(" ", (RESULTS_DIR / "feature_types.csv").relative_to(PROJECT_ROOT))

Saved:
  data\processed\train_features.parquet
  data\processed\test_features.parquet
  results\feature_types.csv


## 3. Layer B 

Evaluating preprocessing choices on the same CV splits used by the
baseline, just for a reference

The preprocessor is created with `make_preprocessor(...)` and wrapped in a
`Pipeline`, so `evaluate_model` clones and refits it inside every fold.

In [5]:
labeled = train_features[train_features[TARGET].notna()].reset_index(drop=True)
feature_columns = [c for c in labeled.columns if c not in {ID_COLUMN, TARGET}]

X = labeled[feature_columns].copy()
y = labeled[TARGET].astype(int)
ids = labeled[ID_COLUMN]

cv_splits = create_cv_splits(X=X, y=y, ids=ids)
print("X:", X.shape, "| features:", len(feature_columns))

X: (2736, 65) | features: 65


In [6]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge

print("Ridge | Layer A + median imputation")
ridge_median = Pipeline([
    ("preprocessor", make_preprocessor(X, strategy="median")),
    ("model", Ridge(alpha=1.0)),
])
result_median = evaluate_model(
    ridge_median, X, y, cv_splits, prediction_transform=regression_to_classes,
)

Ridge | Layer A + median imputation


Fold 1: training QWK=0.4001, validation QWK=0.3474
Fold 2: training QWK=0.3790, validation QWK=0.4249


Fold 3: training QWK=0.4033, validation QWK=0.3640
Fold 4: training QWK=0.4036, validation QWK=0.3586


Fold 5: training QWK=0.4097, validation QWK=0.3544
Mean training QWK: 0.3992
Mean validation QWK: 0.3698
Validation QWK standard deviation: 0.0280


In [7]:
print("Ridge | Layer A + iterative block imputation (BIA + Physical)")
ridge_iterative = Pipeline([
    ("preprocessor", make_preprocessor(X, strategy="iterative")),
    ("model", Ridge(alpha=1.0)),
])
result_iterative = evaluate_model(
    ridge_iterative, X, y, cv_splits, prediction_transform=regression_to_classes,
)

Ridge | Layer A + iterative block imputation (BIA + Physical)


Fold 1: training QWK=0.4046, validation QWK=0.3460


Fold 2: training QWK=0.3870, validation QWK=0.4128


Fold 3: training QWK=0.3991, validation QWK=0.3795


C:\Python\Anaconda\Lib\site-packages\sklearn\impute\_iterative.py:825: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


Fold 4: training QWK=0.4120, validation QWK=0.3483


Fold 5: training QWK=0.4180, validation QWK=0.3586
Mean training QWK: 0.4041
Mean validation QWK: 0.3691
Validation QWK standard deviation: 0.0249


In [8]:
# Fold-safe outlier handling for the scale-sensitive linear model:
# clip numeric features to the [1%, 99%] percentiles, fit per fold.
print("Ridge | Layer A + median + winsorize 1%")
ridge_winsor = Pipeline([
    ("preprocessor", make_preprocessor(X, strategy="median", winsorize=0.01)),
    ("model", Ridge(alpha=1.0)),
])
result_winsor = evaluate_model(
    ridge_winsor, X, y, cv_splits, prediction_transform=regression_to_classes,
)

Ridge | Layer A + median + winsorize 1%
Fold 1: training QWK=0.4097, validation QWK=0.3385
Fold 2: training QWK=0.3946, validation QWK=0.4290
Fold 3: training QWK=0.4031, validation QWK=0.3670


Fold 4: training QWK=0.4123, validation QWK=0.3555
Fold 5: training QWK=0.4169, validation QWK=0.3594
Mean training QWK: 0.4073
Mean validation QWK: 0.3699
Validation QWK standard deviation: 0.0310


In [9]:
# Recommended direction: a NaN-native tree model needs no imputation at all.
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.ensemble import HistGradientBoostingClassifier

categorical_columns = X.select_dtypes(include=["object", "string", "category"]).columns.tolist()
native_nan_encoder = ColumnTransformer(
    [("categorical", OrdinalEncoder(
        handle_unknown="use_encoded_value", unknown_value=-1, encoded_missing_value=-2,
    ), categorical_columns)],
    remainder="passthrough",
)

print("HistGradientBoosting | Layer A, native NaN, DEFAULT params (untuned)")
hist_gbm = Pipeline([
    ("encoder", native_nan_encoder),
    ("model", HistGradientBoostingClassifier(random_state=42)),
])
result_hist = evaluate_model(hist_gbm, X, y, cv_splits)

HistGradientBoosting | Layer A, native NaN, DEFAULT params (untuned)


Fold 1: training QWK=1.0000, validation QWK=0.3171


Fold 2: training QWK=0.9992, validation QWK=0.3529


Fold 3: training QWK=1.0000, validation QWK=0.2972


Fold 4: training QWK=1.0000, validation QWK=0.3097


Fold 5: training QWK=1.0000, validation QWK=0.2783
Mean training QWK: 0.9998
Mean validation QWK: 0.3111
Validation QWK standard deviation: 0.0247


In [10]:
summary = pd.DataFrame([
    {
        "setup": name,
        "mean_val_qwk": np.mean(r["validation_scores"]),
        "val_std_qwk": np.std(r["validation_scores"]),
        "mean_train_qwk": np.mean(r["training_scores"]),
    }
    for name, r in [
        ("Ridge | Layer A + median", result_median),
        ("Ridge | Layer A + iterative", result_iterative),
        ("Ridge | Layer A + median + winsor 1%", result_winsor),
        ("HistGBM | Layer A, native NaN (untuned)", result_hist),
    ]
]).round(4)

RESULTS_DIR.mkdir(exist_ok=True)
summary.to_csv(RESULTS_DIR / "imputation_cv_results.csv", index=False)
print("Documented baseline (01/baseline.ipynb): Ridge mean val QWK = 0.3677, std = 0.0386\n")
summary

Documented baseline (01/baseline.ipynb): Ridge mean val QWK = 0.3677, std = 0.0386



,setup,mean_val_qwk,val_std_qwk,mean_train_qwk
0,Ridge | Layer A + median,0.3698,0.0280,0.3992
1,Ridge | Layer A + iterative,0.3691,0.0249,0.4041
2,Ridge | Layer A + median + winsor 1%,0.3699,0.0310,0.4073
3,"HistGBM | Layer A, native NaN (untuned)",0.3111,0.0247,0.9998


## 4. Main findings to carry forward

1. **Validation catches real invalid values**:
   impossible `Physical-Weight`/`BMI`/`BP` = 0 and a `CGAS-CGAS_Score` = 999 sentinel
   are voided to NaN (row-local, dictionary/clinical bounds, so safe to persist).
   Codebook and BMI cross-consistency found no further problems on this data.
2. **Layer A keeps the signal and stabilizes CV.** Ridge with Layer A + median
   reaches the same mean QWK as the raw baseline (~0.369) with lower fold-to-fold
   variance (~0.026 vs 0.039); the gain is stability and a cleaner, NaN-informative
   feature space, not a jump in mean.
3. **Iterative block imputation and winsorization are not adopted by default.**
   On the linear model neither beats plain median, so the simpler, more robust
   median is the default; both remain available options in `make_preprocessor`.
4. **NaN-native trees are the recommended modeling direction** but must be
   regularized/tuned, because untuned HistGBM overfits (train QWK ~1.0).

**Deliverables**

- `data/processed/train_features.parquet`, `data/processed/test_features.parquet`
  (Layer A applied; NaN present; `id`/`sii` kept).
- `results/imputation_cv_results.csv`: reference CV numbers.
- `results/feature_types.csv`: dictionary-driven type contract per field.

**General rule:** call `make_preprocessor(...)` only inside a `Pipeline`/CV. Never
fit it on the full training data and never save its output; for the final test
prediction, fit on all of train and apply to test exactly once.